In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# prompt: face dataset drive zip fiel

# Specify the path to your zip file in Google Drive
zip_file_path = '/content/drive/MyDrive/strawberry.zip' # Replace with the actual path

# Specify the directory where you want to extract the dataset
extract_dir = '/content/stroberry_dataset'

# Create the extraction directory if it doesn't exist
!mkdir -p {extract_dir}

# Unzip the file
!unzip {zip_file_path} -d {extract_dir}

print(f"Dataset extracted to {extract_dir}")

Streaming output truncated to the last 5000 lines.
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot108_jpg.rf.ac9843c0cf425cdf9d89b16432f0cb62.jpg  
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot108_jpg.rf.d2bce9a44e6ac4a4fa8415cd80f7ad16.jpg  
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot109.rf.5294adc2c92336c30ec737675ad0f5c5.jpg  
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot109.rf.5c01439134ea459fb42643ee5e578f95.jpg  
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot109.rf.d3812a0f84d90c14686b0fd3291555f3.jpg  
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot109_jpg.rf.10c62d752c7280fa1781151fc143c7da.jpg  
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot109_jpg.rf.43995b49a266ab1e36de4188ce555465.jpg  
  inflating: /content/stroberry_dataset/Strawberry_spot/leaf_spot109_jpg.rf.5a8ca79d52c5c3d34e7a44a4b0858b60.jpg  
  inflating: /content/stroberry_dataset/S

In [ ]:
!pip install tensorflow matplotlib


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# Load dataset
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "stroberry_dataset",
    image_size=(180, 180),     # Resize images
    batch_size=32              # Number of images per batch
)

# Split into training and validation
train_ds = dataset.take(int(len(dataset)*0.8))
val_ds = dataset.skip(int(len(dataset)*0.8))

# Prefetch to improve performance
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

# Build model
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(180, 180, 3)),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(3)  # 3 categories
])

# Compile
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Train
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

# Save model
model.save("strawberry_classifier.h5")


Found 5164 files belonging to 3 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 15s 68ms/step - accuracy: 0.6230 - loss: 0.6783 - val_accuracy: 0.8668 - val_loss: 0.3024
Epoch 2/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9031 - loss: 0.2398 - val_accuracy: 0.9363 - val_loss: 0.1530
Epoch 3/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.9441 - loss: 0.1471 - val_accuracy: 0.9315 - val_loss: 0.1734
Epoch 4/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.9527 - loss: 0.1167 - val_accuracy: 0.9710 - val_loss: 0.0805
Epoch 5/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.9746 - loss: 0.0687 - val_accuracy: 0.9720 - val_loss: 0.0834
Epoch 6/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.9733 - loss: 0.0751 - val_accuracy: 0.9653 - val_loss: 0.1214
Epoch 7/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 7s 56ms/step - accuracy: 0.9860 - loss: 0.0423 - val_accuracy: 0.9778 - val_loss: 0.0523
Epoch 8/10
129/129 ━━━━━━━━━━━━━━━━━━━━ 10s 75ms/step - accuracy: 0.9936 - loss: 0.0167 - val_ac

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img = image.load_img("/content/stroberry_dataset/Strawberry_fresh/01e591c9-e3e7-4edc-8211-13081f4d5e7a___RS_HL 1979.JPG", target_size=(180, 180))
img_array = image.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)  # Add batch dimension

predictions = model.predict(img_array)
score = tf.nn.softmax(predictions[0])

class_names = dataset.class_names
print(f"This image is {class_names[np.argmax(score)]} with {100 * np.max(score):.2f}% confidence.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 610ms/step
This image is Strawberry_fresh with 98.83% confidence.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

# === 1. Load Dataset with Validation Split ===
batch_size = 32
img_size = (224, 224)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "stroberry_dataset",
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "stroberry_dataset",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)

# === 2. Preprocessing and Augmentation ===
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def preprocess(image, label):
    image = preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess).map(lambda x, y: (data_augmentation(x), y)).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

# === 3. Load Pretrained ResNet50 Base ===
resnet_base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
resnet_base.trainable = False  # First freeze for feature extraction

# === 4. Build the Model ===
model = models.Sequential([
    resnet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes)  # 3 output classes
])

# === 5. Compile & Train (Feature Extraction) ===
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop])

# === 6. Fine-Tune Top ResNet Layers ===
resnet_base.trainable = True
for layer in resnet_base.layers[:-30]:  # Fine-tune last 30 layers
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop])

# === 7. Save the Trained Model ===
model.save("strawberry_resnet50_final.h5")


Found 5164 files belonging to 3 classes.
Using 4132 files for training.
Found 5164 files belonging to 3 classes.
Using 1032 files for validation.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Epoch 1/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 87s 563ms/step - accuracy: 0.8727 - loss: 0.3344 - val_accuracy: 0.9981 - val_loss: 0.0109
Epoch 2/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 58s 447ms/step - accuracy: 0.9902 - loss: 0.0354 - val_accuracy: 0.9981 - val_loss: 0.0064
Epoch 3/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 58s 447ms/step - accuracy: 0.9946 - loss: 0.0161 - val_accuracy: 0.9981 - val_loss: 0.0088
Epoch 4/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 83s 456ms/step - accuracy: 0.9951 - loss: 0.0121 - val_accuracy: 0.9971 - val_loss: 0.0089
Epoch 5/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 58s 446ms/step - accuracy: 0.9960 - loss: 0.0143 - val_accuracy: 0.9981 - val_loss: 0.0045
Epoch 6/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 83s 451ms/step - accuracy: 0.9980 - loss: 0.0096 - val_accuracy: 0.9990 - val_loss: 0.0026
Epoch 7/10
130/13

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img_path = "/content/stroberry_dataset/Strawberry_spot/angular_leafspot169.rf.faeca6bf315f6acc5aad97753e959092.jpg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)
img_array = preprocess_input(img_array)

predictions = model.predict(img_array)
score = tf.nn.softmax(predictions[0])
print(f"Predicted class: {class_names[np.argmax(score)]} ({100 * np.max(score):.2f}% confidence)")



1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Predicted class: Strawberry_spot (99.95% confidence)
